# Agente de Triagem de Ordens de Serviço de Manutenção

## Entrega 1 — Caracterização do Caso e Agente Único

**Engenharia de Agentes e IA Agêntica** · Pós-graduação · PUC Minas · Prof. Paulo Moreira

| | |
|---|---|
| **Empresa (fictícia)** | Construtora Apollo S.A. |
| **Ambiente** | Google Colab |
| **SDK** | `google-genai` · `gemini-3.6-flash` · embeddings `gemini-embedding-001` |
| **Base de conhecimento** | Vector RAG com ChromaDB, local |
| **Ferramentas** | Tool Calling estruturado, integração direta (sem MCP) |
| **Repositório** | [github.com/diogo-hsf/apollo-agente-manutencao](https://github.com/diogo-hsf/apollo-agente-manutencao) |
| **Requisito externo** | Apenas a `GEMINI_API_KEY` |

> **Como executar.** Cadastre a chave do Gemini nos *Secrets* do Colab com o nome `GEMINI_API_KEY` (ícone de chave na barra lateral) e use *Ambiente de execução → Executar tudo*. Documentos e tabelas são baixados do repositório pelo próprio notebook.

> **Sobre os dados.** A Construtora Apollo S.A., seus documentos técnicos, os códigos de alarme TL e os códigos internos de peças APL são fictícios, criados para esta disciplina. Os modelos de equipamento são reais (Caterpillar 320, D9T e 924H; Mercedes-Benz Axor e 2726). Os manuais dos fabricantes foram usados apenas como referência factual — intervalos de manutenção, lógica de níveis de alerta e referências de peças —, sem reprodução de trechos.

## Sumário

1. Caracterização do caso
2. Desenho da solução
3. Configuração do ambiente
4. Dados operacionais
5. Base de conhecimento técnica (Vector RAG)

---
# 1. Caracterização do caso

## 1.1 A empresa

A **Construtora Apollo S.A.** executa obras de infraestrutura pesada — duplicação rodoviária, alteamento de barragem de mineração e terraplenagem industrial — com frota própria de escavadeiras, tratores de esteiras, carregadeiras de rodas e caminhões basculantes, distribuída entre as obras.

Nesse negócio, equipamento parado é custo direto. Uma escavadeira parada não só deixa de produzir: ela trava os caminhões que carregaria, a frente de serviço que dependia dela e o cronograma da obra. O indicador perseguido pela gestão de manutenção é a **disponibilidade física da frota**.

## 1.2 O problema de negócio

Quando um equipamento apresenta problema, o operador abre uma **ordem de serviço (OS)** de manutenção. Ela chega ao planejador de manutenção como um texto curto, escrito no campo e quase sempre incompleto:

- *"Alarme TL-214 no painel, lança e braço ficam lentos depois de umas duas horas de trabalho."*
- *"Caminhão fazendo barulho estranho na frente."*
- *"Vazamento de óleo no cilindro da caçamba da escavadeira."* — sem dizer qual escavadeira.

Antes que um mecânico vá ao equipamento, alguém precisa fazer a **triagem** dessa OS:

| Pergunta da triagem | Onde está a resposta hoje |
|---|---|
| O que esse alarme significa e qual o procedimento de diagnóstico? | Fichas técnicas e catálogo de alarmes, em PDF |
| A situação exige bloquear o equipamento? | Procedimento de segurança, em PDF |
| Essa falha já aconteceu antes nesse equipamento? | Histórico de manutenção, no sistema |
| A revisão preventiva está em dia? | Plano de manutenção e horímetro, no sistema |
| A peça provável está disponível? Em qual almoxarifado? | Estoque, no sistema |
| Há equipamento substituto se este parar? | Cadastro da frota, no sistema |
| Qual a prioridade? | Política de manutenção, cruzada com tudo acima |

Hoje essa triagem é manual. A informação está espalhada entre documentos e sistemas, e a busca leva minutos ou horas por OS. O custo aparece em quatro lugares:

- **Parada prolongada:** o mecânico só começa quando a triagem termina — ou começa sem ela.
- **Diagnóstico errado:** sem consultar o histórico, troca-se a mesma peça pela terceira vez sem atacar a causa raiz.
- **Espera por peça:** a falta da peça só é descoberta com o equipamento desmontado.
- **Risco de segurança:** um equipamento com falha que exige bloqueio continua operando porque ninguém consultou o procedimento a tempo.

## 1.3 Duas naturezas de informação

A triagem combina dois tipos de informação, e a solução trata cada um de forma diferente:

| | **Conhecimento técnico (regras)** | **Estado operacional (fatos)** |
|---|---|---|
| O que é | Significado de alarmes, procedimentos de diagnóstico, regras de segurança e de prioridade | Situação atual de cada equipamento, peça e obra |
| Onde vive | Fichas técnicas, catálogo de alarmes, procedimento de segurança, política de manutenção | Cadastro, histórico, horímetro e preventivas, estoque, frota |
| Muda com que frequência | Raramente, por revisão de documento | A cada OS, a cada movimentação de estoque |
| Exemplo | *"Três corretivas no mesmo sistema em 90 dias caracterizam falha recorrente"* | *"A EH-07 teve duas corretivas hidráulicas nos últimos 45 dias"* |
| Como o agente acessa | **Base de conhecimento (Vector RAG)** | **Ferramentas (Tool Calling)** |

## 1.4 O que vamos construir

Um **agente de triagem de OS de manutenção**. Para cada OS aberta, o agente investiga — consultando a documentação técnica e os sistemas conforme a necessidade *daquela* OS — e registra na **fila de manutenção** um parecer com:

| Saída | Pergunta que responde |
|---|---|
| Diagnóstico provável | O que provavelmente está acontecendo, com a fonte técnica que sustenta |
| Procedimento recomendado | Por onde o mecânico deve começar |
| Peças e disponibilidade | O que provavelmente será necessário e onde existe em estoque |
| Recorrência | A falha é nova ou repetida? |
| Situação da preventiva | A revisão está em dia, próxima ou vencida? |
| Prioridade | Com que urgência tratar, segundo a matriz da política de manutenção |
| Recomendação operacional | Liberar, liberar com restrição ou bloquear — **sempre como recomendação** |
| Pendências e perguntas | O que não foi possível verificar e o que precisa ser perguntado ao solicitante |

## 1.5 O que o agente **não** faz

- **Não executa** reparo nem altera nada nos sistemas de manutenção.
- **Não bloqueia nem libera** equipamentos — apenas recomenda; a decisão é de pessoa habilitada (PRO-SEG-001).
- **Não emite** requisição de compra de peças.
- **Não substitui** a inspeção presencial do mecânico.
- **Não inventa** diagnóstico. Se o relato for vago ou o equipamento não puder ser identificado, a saída correta é devolver a OS com perguntas.

## 1.6 Por que uma abordagem agêntica?

Pelo critério visto na Aula 2 — *"o fluxo é conhecido e determinável? Então um workflow pode bastar"* —, a resposta depende de o caminho da triagem ser previsível. Ele não é.

**O caminho da investigação muda a cada OS.** Quatro ordens do mesmo dia exigem caminhos diferentes:

| OS | Caminho de investigação adequado |
|---|---|
| *"EH-07 com alarme TL-214, lança e braço lentos depois de aquecer"* | Ficha técnica (TL-214) → histórico (a falha se repete?) → política (o que fazer com falha recorrente) → estoque |
| *"Caminhão fazendo barulho estranho na frente"* | Ficha técnica mostra que o ruído pode ser freio, suspensão ou direção → relato insuficiente → devolver com as perguntas certas |
| *"Trator perdendo força na rampa, fumaça escura no escapamento"* | Preventiva primeiro: revisão de filtros vencida é a hipótese principal; histórico confirma |
| *"Vazamento de óleo no cilindro da caçamba da escavadeira"* | Equipamento não identificado e há mais de uma escavadeira na obra → não é permitido deduzir → devolver |

Um workflow determinístico teria de prever cada combinação de sintoma, equipamento e situação — ou executaria **todas** as consultas em **toda** OS, gerando custo, ruído e respostas genéricas.

**A entrada é texto livre e incompleto.** O sintoma vem na linguagem do operador, com ou sem código de alarme. Interpretar *"fica lento depois que esquenta"* exige julgamento, não regra.

**É preciso cruzar regra e fato.** *"A ficha manda limpar o arrefecedor e trocar o filtro"* (regra) somado a *"isso já foi feito duas vezes neste mês"* (fato) muda a conclusão para *"falha recorrente: análise de causa raiz; repetir a troca não resolve"*.

**É preciso saber quando parar — e quando não concluir.** O agente decide quando já tem evidência suficiente e reconhece quando a informação disponível não sustenta um diagnóstico.

**O que continua determinístico.** Seguindo o princípio *aritmética no código, julgamento no modelo*, ficam em Python: horas desde a última preventiva, contagem de recorrência, saldo de estoque, validação de identificadores, detecção de gatilhos de segurança, limites de execução e validação do parecer.

**Por que um agente único.** A triagem é uma responsabilidade coesa, com um objetivo e um responsável. Ainda não há especialização que justifique dividir o trabalho entre vários agentes.

## 1.7 O ambiente do agente

```
═══════════════════════════ AMBIENTE ═══════════════════════════

              Operador / motorista do equipamento
                         │  abre uma OS
                         ▼
                ordens_servico.csv
                         │  percepção
                         ▼
   ┌───────────────────────────────────────────────────┐
   │        AGENTE DE TRIAGEM DE MANUTENÇÃO            │
   │                                                   │
   │   perceber()  → o que foi relatado?               │
   │   decidir()   → laço: decidir → agir → observar   │
   │   agir()      → registrar o parecer               │
   │                                                   │
   │   ┌───────────────────────────────────────────┐   │
   │   │ LLM (gemini-3.6-flash)                    │   │
   │   │ mecanismo de inferência                   │   │
   │   └───────────────────────────────────────────┘   │
   └───────────────────────────────────────────────────┘
          │                         ▲
          │ ação                    │ consultas (Tool Calling)
          ▼                         │
  fila_manutencao.csv     ┌─────────┴──────────────────────────┐
          │               │ Documentação técnica (Vector RAG)  │
          ▼               │ Equipamentos · Histórico           │
  Planejador de           │ Preventivas · Estoque · Frota      │
  manutenção / mecânico   └────────────────────────────────────┘

════════════════════════════════════════════════════════════════
```

## 1.8 Framework PEAS

| Elemento | Neste agente |
|---|---|
| **P**erformance | Diagnóstico fundamentado em fonte técnica recuperada; nenhum fato operacional afirmado sem consulta ao sistema; ferramentas escolhidas de acordo com a OS; devolução de OS com informação insuficiente; recomendação de segurança conservadora; conclusão dentro dos limites de execução |
| **E**nvironment | Ordens de serviço da Construtora Apollo, documentação técnica da frota e sistemas de manutenção, estoque e frota |
| **A**ctuators | Registro do parecer na fila de manutenção (`fila_manutencao.csv`) |
| **S**ensors | Leitura da OS e resultados das ferramentas: documentação técnica, cadastro, histórico, preventivas, estoque e frota |

Os **sensores são ativos**: o agente não apenas recebe a OS — ele decide o que mais precisa observar.

## 1.9 Propriedades do ambiente

| Propriedade | Classificação | Consequência para o projeto |
|---|---|---|
| Observável | **Parcialmente** | O relato é incompleto e o agente não vê a máquina; precisa de ferramentas e deve declarar incertezas |
| Agentes | **Único** | Só este agente atua; os humanos consomem o resultado |
| Determinismo | **Estocástico** | A mesma falha pode ter causas diferentes; o diagnóstico é sempre "provável" |
| Episódico | **Episódico por OS, com memória no histórico** | Cada OS é independente, mas o histórico de OS anteriores é informação decisiva |
| Estático | **Dinâmico** | Estoque e status mudam ao longo do dia; fatos são consultados no momento da triagem, nunca supostos |
| Conhecido | **Conhecido** | As regras estão documentadas; o desafio é localizá-las e aplicá-las |

## 1.10 Tipo de agente e nível de autonomia

**Tipo.** Agente **baseado em objetivo** — tem uma meta explícita (uma triagem fundamentada e acionável) e escolhe uma sequência de ações para atingi-la —, com raciocínio no padrão **ReAct**: decide, age (consulta), observa o resultado e decide de novo. A definição de prioridade tem um componente de **utilidade**, porque pondera segurança, criticidade e disponibilidade de substituto. A detecção de gatilhos de segurança é uma **camada reativa** em código, o que caracteriza um agente **híbrido**.

| Característica | Presente? | Por quê |
|---|---|---|
| Autonomia | Parcial | Decide sozinho o que investigar e o que concluir, mas apenas recomenda |
| Reatividade | Sim | É acionado pela abertura de uma OS |
| Proatividade | Parcial | Dentro da triagem, busca por iniciativa própria o que falta |
| Sociabilidade | Não | Agente único nesta entrega |

**Nível de autonomia** (matriz *"qual nível para cada tarefa"*, Aula 1):

| Decisão | Impacto | Nível adotado |
|---|---|---|
| Diagnóstico, procedimento, peças, prioridade | Médio e reversível — o mecânico confirma em campo | **Supervisionado**: o agente produz, o planejador revisa |
| Liberar ou bloquear o equipamento | Alto — segurança de pessoas | **Humano no loop, sempre**: o agente só recomenda, com viés conservador |

## 1.11 Escopo desta entrega

| Esta entrega contempla | Fica para as próximas |
|---|---|
| Agente único com laço de decisão, ação e observação (ReAct) | Orquestração com framework |
| Base de conhecimento técnica com Vector RAG | Sistema multiagente |
| Seis ferramentas com Tool Calling estruturado | Governança, auditoria e human-in-the-loop formal |
| Validação, tratamento de erros, limites e recuperação de falhas | Memória entre triagens |
| Testes de comportamento | |

---
# 2. Desenho da solução

## 2.1 Arquitetura

```
                 ordens_servico.csv
                         │
                         ▼
┌──────────────────────────────────────────────────────────────┐
│ perceber()                                     [código]      │
│ valida a OS · extrai códigos de alarme · resolve o           │
│ equipamento no cadastro · detecta gatilhos de segurança      │
└──────────────────────────────────────────────────────────────┘
                         │
                         ▼
┌──────────────────────────────────────────────────────────────┐
│ decidir()  laço ReAct                     [LLM + runtime]    │
│                                                              │
│  ┌──────────┐   function_call    ┌───────────────────────┐   │
│  │  Gemini  │ ─────────────────► │ Runtime               │   │
│  │ 3.6 Flash│                    │ · valida argumentos   │   │
│  │          │ ◄───────────────── │ · aplica limites      │   │
│  └──────────┘ function_response  │ · executa a tool      │   │
│        │                         │ · normaliza erros     │   │
│        │ emitir_parecer(...)     └───────────┬───────────┘   │
│        ▼                                     │               │
│  ┌──────────────────────────┐   ┌────────────▼────────────┐  │
│  │ Validação do parecer     │   │ Ferramentas             │  │
│  │ schema · domínio ·       │   │ RAG técnico · histórico │  │
│  │ guardrail de segurança   │   │ preventiva · estoque ·  │  │
│  └──────────────────────────┘   │ frota · equipamento     │  │
│                                 └─────────────────────────┘  │
└──────────────────────────────────────────────────────────────┘
                         │
                         ▼
┌──────────────────────────────────────────────────────────────┐
│ agir()                                         [código]      │
│ grava o parecer em fila_manutencao.csv                       │
└──────────────────────────────────────────────────────────────┘
```

Sem MCP, o notebook usa o cliente **síncrono** do `google-genai` e um **registro explícito** de ferramentas (`nome → função + schema de entrada`), como no slide de integração direta da Aula 3.

## 2.2 Divisão de responsabilidades

| Em Python (determinístico) | No modelo (julgamento) |
|---|---|
| Validar a OS e resolver o equipamento informado | Interpretar o relato em linguagem livre |
| Extrair códigos de alarme do texto (`TL-214`) | Decidir o que investigar e em que ordem |
| Calcular horas desde a última preventiva e a situação | Formular as perguntas à base técnica |
| Contar ocorrências do mesmo sistema em 90 dias | Cruzar regra técnica com fato operacional |
| Somar saldo de estoque por almoxarifado | Formar diagnóstico provável e prioridade |
| Detectar gatilhos críticos de segurança no relato | Decidir quando parar ou devolver a OS |
| Aplicar limites e validar o parecer | |

## 2.3 Base de conhecimento técnica

| Código | Documento | Aplicação |
|---|---|---|
| FTM-CAT320 | Ficha técnica de manutenção — Escavadeira CAT 320 | escavadeiras |
| FTM-CATD9T | Ficha técnica de manutenção — Trator de esteiras CAT D9T | tratores |
| FTM-CAT924H | Ficha técnica de manutenção — Carregadeira CAT 924H | carregadeiras |
| FTM-MBCAM | Ficha técnica de manutenção — Caminhões basculantes Mercedes-Benz | caminhões |
| CAT-ALM-001 | Catálogo de alarmes de telemetria (códigos TL) | geral |
| PRO-SEG-001 | Procedimento de segurança — bloqueio e impedimento de operação | geral |
| POL-MAN-001 | Política de manutenção da frota | geral |

Os PDFs têm seções numeradas, o que permite **chunking por seção** e citações no formato `FTM-CAT924H § 5.3`. Cada chunk recebe nos metadados a sua aplicação: a busca para uma escavadeira traz a ficha da escavadeira e os documentos gerais, nunca a ficha do caminhão.

A matriz de prioridade e a regra de recorrência ficam **no documento**, não no código: o agente precisa recuperar a regra antes de aplicá-la — o padrão "regra no documento, fato no sistema".

## 2.4 Ferramentas

| Ferramenta | Pergunta que responde | Fonte |
|---|---|---|
| `buscar_documentacao_tecnica` | O que diz a documentação? | Vector RAG |
| `consultar_equipamento` | Qual equipamento é este? | Cadastro |
| `consultar_historico_manutencao` | Já aconteceu antes? | Histórico de corretivas |
| `consultar_plano_preventiva` | A manutenção está em dia? | Plano + registro de preventivas |
| `consultar_estoque_pecas` | A peça existe? Onde? | Estoque |
| `consultar_frota_disponivel` | Há substituto se parar? | Cadastro |
| **`emitir_parecer`** | *(ferramenta terminal)* | — |

Todas as ferramentas devolvem um **status padronizado**: `success`, `not_found` (consulta válida, sem registro — ausência de informação, não erro), `invalid_arguments` (com a mensagem de correção) ou `error` (falha interna, sem detalhes técnicos expostos ao modelo).

**Ferramenta terminal.** O parecer final é entregue chamando `emitir_parecer`, cujos argumentos seguem o schema da saída. Isso dispensa uma chamada extra de finalização: o modelo conclui na mesma rodada em que decide parar.

## 2.5 Limites e robustez

| Mecanismo | Como funciona |
|---|---|
| Limite de rodadas do modelo e de chamadas de ferramenta por triagem | Ao atingir, o runtime força a finalização: só `emitir_parecer` fica disponível |
| Chamada repetida | A mesma ferramenta com os mesmos argumentos não é executada de novo |
| Validação de entrada | Argumentos conferidos contra o schema antes de executar |
| Validação de retorno | Saída da ferramenta conferida contra o contrato antes de chegar ao modelo |
| Erros de API transitórios (429, 500, 503, 504) | Retry com backoff exponencial |
| Cota diária esgotada | Sem retry inútil: triagem marcada `incompleta`, evidências preservadas |
| Parecer inválido | Uma tentativa de reparo com os erros apontados; persistindo, revisão humana obrigatória |
| Gatilho de segurança | Camada reativa em código impede a recomendação "liberar" |

## 2.6 Orçamento de chamadas

O free tier do Gemini permite cerca de **20 chamadas de geração por dia** por modelo. O notebook conta cada chamada e se recusa a ultrapassar um orçamento global, para que uma execução completa — demonstração e testes — caiba nesse limite. Os testes de robustez usam um **modelo simulado**, sem custo de cota.

---
# 3. Configuração do ambiente

## 3.1 Dependências

Versões fixadas para que o notebook se comporte igual em qualquer execução. O `google-genai` segue a mesma versão usada nas aulas.

In [ ]:
%pip install -q "google-genai==2.25.0" "chromadb>=1.0,<2" "pypdf>=5,<7" "jsonschema>=4.23,<5"

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import random
import re
import time
import unicodedata
import urllib.request
from dataclasses import dataclass, field
from datetime import date, datetime, timedelta
from functools import lru_cache
from pathlib import Path
from typing import Any, Callable

import chromadb
import pandas as pd
from google import genai
from google.genai import errors, types
from jsonschema import Draft202012Validator
from pypdf import PdfReader

print("Bibliotecas importadas.")
print(f"google-genai {genai.__version__} · chromadb {chromadb.__version__} · pandas {pd.__version__}")

## 3.2 Parâmetros

Todos os parâmetros que mudam o comportamento do notebook ficam nesta célula.

`DATA_REFERENCIA` é o "hoje" do caso. As janelas de histórico e o cálculo de recorrência usam essa data, e não a data real de execução — assim o resultado é o mesmo em qualquer dia em que o notebook for executado.

In [ ]:
# Repositório com documentos e tabelas (público; baixado via raw.githubusercontent.com)
REPOSITORIO_RAW = "https://raw.githubusercontent.com/diogo-hsf/apollo-agente-manutencao/main/"
PASTA_LOCAL = Path("apollo_dados")            # cópia local dos arquivos baixados

DATA_REFERENCIA = date(2026, 9, 28)           # "hoje" do caso

# Modelos
MODELO = "gemini-3.6-flash"
MODELO_EMBEDDING = "gemini-embedding-001"
DIMENSAO_EMBEDDING = 768

# Resiliência das chamadas à API
TENTATIVAS_POR_CHAMADA = 5                    # retry para erros transitórios
ESPERA_BASE_SEGUNDOS = 5.0                    # backoff: 5, 10, 20, 40 s
ESPERA_MAXIMA_ACEITAVEL = 90                  # retryDelay maior que isso = cota diária
TIMEOUT_CHAMADA_MS = 120_000

# Orçamento de chamadas de GERAÇÃO para o notebook inteiro
# (o free tier do gemini-3.6-flash permite ~20 por dia)
ORCAMENTO_CHAMADAS_GERACAO = 18

print(f"Modelo: {MODELO} · embeddings: {MODELO_EMBEDDING} ({DIMENSAO_EMBEDDING} dim)")
print(f"Data de referência do caso: {DATA_REFERENCIA:%d/%m/%Y}")
print(f"Orçamento de chamadas de geração: {ORCAMENTO_CHAMADAS_GERACAO}")

## 3.3 Chave da API e conexão

A chave é lida dos *Secrets* do Colab (`GEMINI_API_KEY`), de uma variável de ambiente ou, em último caso, digitada com entrada oculta. Ela nunca aparece no código.

Dois detalhes de configuração do cliente:

- **O retry interno do SDK é desligado** (`attempts=1`). O retry fica a cargo da camada da seção 3.4, que distingue erro transitório de cota diária esgotada. Com os dois ligados, um erro seria tentado de novo em dobro, gastando tempo à toa.
- **A conexão é verificada sem gastar cota.** `models.get` consulta os metadados do modelo: confirma que a chave é válida e que o modelo está disponível, sem consumir nenhuma das ~20 chamadas de geração do dia.

In [ ]:
def obter_api_key() -> str:
    """Recupera a chave do Gemini sem expô-la no código do notebook."""
    try:
        from google.colab import userdata  # type: ignore

        chave = userdata.get("GEMINI_API_KEY")
        if chave:
            print("Chave obtida dos Secrets do Colab.")
            return chave
    except Exception:
        pass

    chave = os.environ.get("GEMINI_API_KEY")
    if chave:
        print("Chave obtida da variável de ambiente GEMINI_API_KEY.")
        return chave

    from getpass import getpass

    chave = getpass("Cole sua GEMINI_API_KEY (a digitação fica oculta): ").strip()
    if not chave:
        raise RuntimeError(
            "Nenhuma chave de API foi fornecida. Gere uma em https://aistudio.google.com/apikey"
        )
    return chave


cliente_genai = genai.Client(
    api_key=obter_api_key(),
    http_options=types.HttpOptions(
        timeout=TIMEOUT_CHAMADA_MS,
        retry_options=types.HttpRetryOptions(attempts=1),  # retry fica na seção 3.4
    ),
)

try:
    info_modelo = cliente_genai.models.get(model=MODELO)
    print(f"Conexão verificada: {info_modelo.display_name or MODELO} disponível (sem consumo de cota).")
except errors.APIError as erro:
    raise RuntimeError(
        f"Não foi possível acessar o modelo '{MODELO}' (HTTP {erro.code}). "
        "Confira se a GEMINI_API_KEY é válida e se o modelo está disponível para ela."
    ) from erro

## 3.4 Camada de acesso ao modelo

Toda chamada ao Gemini — geração ou embedding — passa por `ClienteLLM`. É aqui que mora a robustez contra falhas da API, observada na prática durante o desenvolvimento:

| Situação | Detecção | Resposta |
|---|---|---|
| Sobrecarga do modelo (503) ou erro interno (500, 502, 504) | Código HTTP | Nova tentativa com backoff exponencial: 5, 10, 20, 40 s |
| Limite por minuto (429 com `retryDelay` curto) | `retryDelay` sugerido pela API | Espera o tempo sugerido e tenta de novo |
| Cota diária esgotada (429 com `retryDelay` de horas) | `retryDelay` maior que `ESPERA_MAXIMA_ACEITAVEL` | **Não** tenta de novo: lança `CotaEsgotadaError` na hora |
| Erro do cliente (400, 403...) | Código HTTP | Propaga imediatamente: tentar de novo não corrige argumento inválido |
| Orçamento do notebook atingido | Contador interno | Lança `OrcamentoEsgotadoError` **antes** de chamar a API |

Dois pontos de projeto:

- **A espera é injetável** (`dormir`). Nos testes, um `dormir` falso permite simular cinco erros 503 seguidos sem esperar dois minutos.
- **O agente não conhece o Gemini, conhece o `ClienteLLM`.** Na seção de testes, um modelo simulado implementa a mesma interface (`gerar`), o que permite testar limites e recuperação de falhas de forma determinística e sem gastar cota.

In [ ]:
class CotaEsgotadaError(RuntimeError):
    """Cota do modelo esgotada, sem liberação em tempo útil (ex.: limite diário do free tier)."""


class OrcamentoEsgotadoError(RuntimeError):
    """O notebook atingiu ORCAMENTO_CHAMADAS_GERACAO; a chamada nem chegou a ser feita."""


CODIGOS_TRANSITORIOS = {429, 500, 502, 503, 504}


def _retry_delay_segundos(erro: errors.APIError) -> float | None:
    """Extrai o retryDelay sugerido pela API (ex.: '19144s'), quando presente."""
    texto = json.dumps(getattr(erro, "details", None) or {}, default=str) + " " + str(erro)
    achado = re.search(r"retryDelay\W+(\d+(?:\.\d+)?)s", texto)
    return float(achado.group(1)) if achado else None


def chamar_com_retry(
    funcao: Callable[[], Any],
    *,
    descricao: str,
    tentativas: int = TENTATIVAS_POR_CHAMADA,
    espera_base: float = ESPERA_BASE_SEGUNDOS,
    dormir: Callable[[float], None] = time.sleep,
) -> Any:
    """Executa `funcao` com retry e backoff exponencial para erros transitórios da API."""
    for tentativa in range(1, tentativas + 1):
        try:
            return funcao()
        except errors.APIError as erro:
            codigo = getattr(erro, "code", None)
            if codigo not in CODIGOS_TRANSITORIOS:
                raise
            sugerido = _retry_delay_segundos(erro)
            if codigo == 429 and sugerido and sugerido > ESPERA_MAXIMA_ACEITAVEL:
                raise CotaEsgotadaError(
                    f"Cota esgotada em '{descricao}': a API pede {sugerido / 3600:.1f} h de espera."
                ) from erro
            if tentativa == tentativas:
                raise
            espera = (sugerido or espera_base * 2 ** (tentativa - 1)) + random.uniform(0, 1)
            print(f"  [{descricao}] API respondeu {codigo} — tentativa {tentativa}/{tentativas}; "
                  f"nova tentativa em {espera:.0f}s")
            dormir(espera)
    raise AssertionError("inalcançável")


class ClienteLLM:
    """Ponto único de acesso ao Gemini: retry, cota, orçamento e contabilidade de chamadas."""

    def __init__(self, cliente, modelo: str, modelo_embedding: str,
                 orcamento_geracao: int, dormir: Callable[[float], None] = time.sleep):
        self._cliente = cliente
        self.modelo = modelo
        self.modelo_embedding = modelo_embedding
        self.orcamento_geracao = orcamento_geracao
        self._dormir = dormir
        self.chamadas_geracao = 0
        self.chamadas_embedding = 0

    @property
    def restantes(self) -> int:
        return self.orcamento_geracao - self.chamadas_geracao

    def gerar(self, contents: list[types.Content],
              config: types.GenerateContentConfig) -> types.GenerateContentResponse:
        """Uma chamada de geração (uma rodada do agente). Conta no orçamento."""
        if self.chamadas_geracao >= self.orcamento_geracao:
            raise OrcamentoEsgotadoError(
                f"Orçamento de {self.orcamento_geracao} chamadas de geração atingido."
            )
        self.chamadas_geracao += 1
        return chamar_com_retry(
            lambda: self._cliente.models.generate_content(
                model=self.modelo, contents=contents, config=config
            ),
            descricao="geração",
            dormir=self._dormir,
        )

    def embed(self, textos: list[str], tipo: str, lote: int = 100) -> list[list[float]]:
        """Embeddings em lotes. tipo: 'RETRIEVAL_DOCUMENT' ou 'RETRIEVAL_QUERY'."""
        vetores: list[list[float]] = []
        for inicio in range(0, len(textos), lote):
            parte = textos[inicio:inicio + lote]
            resposta = chamar_com_retry(
                lambda: self._cliente.models.embed_content(
                    model=self.modelo_embedding,
                    contents=parte,
                    config=types.EmbedContentConfig(
                        task_type=tipo, output_dimensionality=DIMENSAO_EMBEDDING
                    ),
                ),
                descricao="embedding",
                dormir=self._dormir,
            )
            self.chamadas_embedding += 1
            vetores.extend(list(e.values) for e in resposta.embeddings)
        return vetores

    def resumo(self) -> str:
        return (f"chamadas de geração: {self.chamadas_geracao}/{self.orcamento_geracao} "
                f"(restam {self.restantes}) · requisições de embedding: {self.chamadas_embedding}")


llm = ClienteLLM(cliente_genai, MODELO, MODELO_EMBEDDING, ORCAMENTO_CHAMADAS_GERACAO)
print("ClienteLLM pronto —", llm.resumo())

---
# 4. Dados operacionais

As tabelas representam os sistemas da Apollo: cadastro da frota, ordens de serviço, histórico de corretivas, plano e registro de preventivas e estoque. Elas são baixadas do repositório e **validadas antes de qualquer uso** — a primeira camada de validação de entradas do sistema.

## 4.1 Download com fallback

`baixar_arquivo` tenta o download três vezes. Se o GitHub estiver indisponível, usa a cópia local de uma execução anterior, se existir. Isso também permite rodar o notebook sem internet: basta colocar os arquivos em `apollo_dados/`, na mesma estrutura do repositório.

In [ ]:
def baixar_arquivo(caminho_relativo: str, tentativas: int = 3) -> Path:
    """Baixa um arquivo do repositório; em falha, recorre à cópia local, se houver."""
    destino = PASTA_LOCAL / caminho_relativo
    destino.parent.mkdir(parents=True, exist_ok=True)
    url = REPOSITORIO_RAW + caminho_relativo
    ultimo_erro: Exception | None = None

    for tentativa in range(1, tentativas + 1):
        try:
            with urllib.request.urlopen(url, timeout=30) as resposta:
                conteudo = resposta.read()
            if not conteudo:
                raise ValueError("arquivo vazio")
            destino.write_bytes(conteudo)
            return destino
        except Exception as erro:  # rede, HTTP 404/5xx, timeout
            ultimo_erro = erro
            if tentativa < tentativas:
                time.sleep(2 * tentativa)

    if destino.exists() and destino.stat().st_size > 0:
        print(f"  aviso: download de {caminho_relativo} falhou ({ultimo_erro}); usando cópia local.")
        return destino
    raise RuntimeError(f"Não foi possível obter {caminho_relativo} ({url}): {ultimo_erro}")

## 4.2 Carga e validação

Cada tabela tem um contrato: colunas obrigatórias, colunas numéricas e chave única. Depois da carga, a integridade entre tabelas também é conferida — por exemplo, todo equipamento precisa pertencer a uma obra cadastrada. Dado inconsistente interrompe o notebook aqui, com mensagem clara, em vez de produzir uma triagem errada mais adiante.

In [ ]:
class DadosInvalidosError(ValueError):
    """Tabela fora do contrato esperado."""


CONTRATO_TABELAS: dict[str, dict[str, list[str]]] = {
    "obras": {
        "colunas": ["obra_id", "nome", "municipio_uf", "almoxarifado_id"],
        "numericas": [], "chave": ["obra_id"]},
    "equipamentos": {
        "colunas": ["equipamento_id", "tipo", "fabricante", "modelo", "ano_fabricacao", "obra_id",
                    "status_operacional", "criticidade", "horimetro_atual", "ficha_tecnica"],
        "numericas": ["ano_fabricacao", "horimetro_atual"], "chave": ["equipamento_id"]},
    "ordens_servico": {
        "colunas": ["os_id", "data_abertura", "obra_id", "equipamento_informado", "aberta_por", "relato"],
        "numericas": [], "chave": ["os_id"]},
    "historico_manutencao": {
        "colunas": ["ordem_id", "equipamento_id", "data_abertura", "data_conclusao", "tipo_manutencao",
                    "sistema", "codigo_alarme", "descricao_servico", "pecas_utilizadas", "horas_parado"],
        "numericas": [], "chave": ["ordem_id"]},
    "plano_preventiva": {
        "colunas": ["ficha_tecnica", "revisao", "intervalo_horas"],
        "numericas": ["intervalo_horas"], "chave": ["ficha_tecnica", "revisao"]},
    "registro_preventivas": {
        "colunas": ["equipamento_id", "revisao", "horimetro_execucao", "data_execucao"],
        "numericas": ["horimetro_execucao"], "chave": ["equipamento_id", "revisao"]},
    "estoque_pecas": {
        "colunas": ["codigo_peca", "descricao", "ref_fabricante", "ficha_tecnica", "almoxarifado_id",
                    "saldo", "unidade", "prazo_reposicao_dias"],
        "numericas": ["saldo", "prazo_reposicao_dias"], "chave": ["codigo_peca", "almoxarifado_id"]},
}


def carregar_tabela(nome: str) -> pd.DataFrame:
    """Baixa e valida uma tabela contra o seu contrato."""
    contrato = CONTRATO_TABELAS[nome]
    df = pd.read_csv(baixar_arquivo(f"dados/{nome}.csv"), dtype=str,
                     keep_default_na=False, encoding="utf-8")

    faltando = [c for c in contrato["colunas"] if c not in df.columns]
    if faltando:
        raise DadosInvalidosError(f"{nome}: colunas ausentes {faltando}")

    for coluna in contrato["numericas"]:
        convertida = pd.to_numeric(df[coluna], errors="coerce")
        invalidas = df.loc[convertida.isna(), coluna].tolist()
        if invalidas:
            raise DadosInvalidosError(f"{nome}.{coluna}: valores não numéricos {invalidas[:3]}")
        df[coluna] = convertida.astype(int)

    duplicadas = df[df.duplicated(subset=contrato["chave"], keep=False)]
    if not duplicadas.empty:
        raise DadosInvalidosError(f"{nome}: chave {contrato['chave']} duplicada em {len(duplicadas)} linhas")
    return df


def validar_integridade(t: dict[str, pd.DataFrame]) -> None:
    """Confere as referências entre tabelas."""
    verificacoes = [
        ("equipamentos.obra_id", t["equipamentos"]["obra_id"], t["obras"]["obra_id"]),
        ("equipamentos.ficha_tecnica", t["equipamentos"]["ficha_tecnica"], t["plano_preventiva"]["ficha_tecnica"]),
        ("historico.equipamento_id", t["historico_manutencao"]["equipamento_id"], t["equipamentos"]["equipamento_id"]),
        ("registro_preventivas.equipamento_id", t["registro_preventivas"]["equipamento_id"], t["equipamentos"]["equipamento_id"]),
        ("ordens_servico.obra_id", t["ordens_servico"]["obra_id"], t["obras"]["obra_id"]),
    ]
    for rotulo, valores, validos in verificacoes:
        orfaos = sorted(set(valores) - set(validos))
        if orfaos:
            raise DadosInvalidosError(f"{rotulo}: referências inexistentes {orfaos}")
    # ordens_servico.equipamento_informado NÃO é verificado de propósito:
    # o que o operador digita pode estar errado — tratar isso é tarefa do agente.


TABELAS = {nome: carregar_tabela(nome) for nome in CONTRATO_TABELAS}
validar_integridade(TABELAS)

print("Tabelas carregadas e validadas:")
for nome, df in TABELAS.items():
    print(f"  {nome:<22} {len(df):>3} linhas · {len(df.columns):>2} colunas")

## 4.3 Visão rápida dos dados

A frota e as ordens de serviço que serão triadas. Repare que a OS-2026-0103 informa um equipamento (`EH-15`) que não existe no cadastro, e que a OS-2026-0106 chegou sem relato — situações que o agente precisa tratar.

In [ ]:
display(TABELAS["equipamentos"][["equipamento_id", "tipo", "modelo", "obra_id",
                                   "status_operacional", "criticidade", "horimetro_atual"]])

In [ ]:
with pd.option_context("display.max_colwidth", 110):
    display(TABELAS["ordens_servico"][["os_id", "obra_id", "equipamento_informado", "relato"]])

---
# 5. Base de conhecimento técnica (Vector RAG)

## 5.1 Documentos

Sete documentos técnicos da Apollo, em PDF. O código do documento é o identificador usado nas citações do parecer.

In [ ]:
DOCUMENTOS_BASE: dict[str, str] = {
    "FTM-CAT320": "Ficha Técnica de Manutenção — Escavadeira Hidráulica CAT 320",
    "FTM-CATD9T": "Ficha Técnica de Manutenção — Trator de Esteiras CAT D9T",
    "FTM-CAT924H": "Ficha Técnica de Manutenção — Carregadeira de Rodas CAT 924H",
    "FTM-MBCAM": "Ficha Técnica de Manutenção — Caminhões Basculantes Mercedes-Benz",
    "CAT-ALM-001": "Catálogo de Alarmes de Telemetria e Ocorrências",
    "PRO-SEG-001": "Procedimento de Segurança — Bloqueio e Impedimento de Operação",
    "POL-MAN-001": "Política de Manutenção da Frota de Equipamentos",
}

ARQUIVOS_PDF = {codigo: baixar_arquivo(f"documentos/{codigo}.pdf") for codigo in DOCUMENTOS_BASE}
for codigo, caminho in ARQUIVOS_PDF.items():
    print(f"  {codigo:<12} {len(PdfReader(caminho).pages)} página(s) · {DOCUMENTOS_BASE[codigo]}")

## 5.2 Chunking por seção

Os documentos têm seções numeradas (`5 SISTEMA DE FREIOS`) e subseções (`5.3 Conduta e diagnóstico`). Cada seção ou subseção vira um chunk, o que traz três vantagens sobre o chunking por número fixo de caracteres:

- **O chunk é uma unidade de sentido.** Um procedimento não é cortado no meio.
- **A citação é precisa.** O agente cita `FTM-CAT924H § 5.3`, e o planejador encontra o trecho na hora.
- **Os metadados permitem filtrar.** Cada chunk carrega a sua `aplicacao`: o código da ficha técnica, ou `geral` para catálogo, procedimento de segurança e política.

Na limpeza, cabeçalhos e rodapés de página são descartados. Seções muito longas seriam subdivididas por linhas, mas nenhuma seção atual chega a esse limite.

In [ ]:
RE_SECAO = re.compile(r"^(\d{1,2}) ([A-ZÁÉÍÓÚÂÊÔÃÕÇ][A-ZÁÉÍÓÚÂÊÔÃÕÇ ,—\-]+)$")
RE_SUBSECAO = re.compile(r"^(\d{1,2}\.\d{1,2}) ([A-ZÁÉÍÓÚÂÊÔÃÕÇ].*)$")
RE_RUIDO = re.compile(r"^(Construtora Apollo S\.A\. · .*|Página \d+)$")
TAMANHO_MAXIMO_CHUNK = 1800


def _linhas_do_pdf(caminho: Path) -> list[str]:
    """Texto do PDF em linhas, sem cabeçalhos e rodapés de página."""
    linhas = []
    for pagina in PdfReader(caminho).pages:
        for linha in (pagina.extract_text() or "").splitlines():
            linha = linha.strip()
            if linha and not RE_RUIDO.match(linha):
                linhas.append(linha)
    return linhas


def _dividir(texto: str, limite: int) -> list[str]:
    """Divide um texto longo em partes, respeitando quebras de linha."""
    if len(texto) <= limite:
        return [texto]
    partes, atual = [], ""
    for linha in texto.split("\n"):
        if atual and len(atual) + len(linha) + 1 > limite:
            partes.append(atual)
            atual = ""
        atual = f"{atual}\n{linha}" if atual else linha
    return partes + ([atual] if atual else [])


def segmentar_documento(codigo: str, caminho: Path) -> list[dict[str, Any]]:
    """Transforma um PDF em chunks por seção, com metadados para filtro e citação."""
    secoes: list[dict[str, Any]] = []
    titulo_pai = ""
    for linha in _linhas_do_pdf(caminho):
        m_secao, m_sub = RE_SECAO.match(linha), RE_SUBSECAO.match(linha)
        if m_secao:
            titulo_pai = m_secao.group(2).capitalize()
            secoes.append({"secao": m_secao.group(1), "titulo": titulo_pai, "linhas": []})
        elif m_sub:
            secoes.append({"secao": m_sub.group(1),
                           "titulo": f"{titulo_pai} › {m_sub.group(2)}", "linhas": []})
        elif secoes:                       # descarta o preâmbulo antes da seção 1
            secoes[-1]["linhas"].append(linha)

    aplicacao = codigo if codigo.startswith("FTM-") else "geral"
    chunks = []
    for secao in secoes:
        texto = "\n".join(secao["linhas"]).strip()
        if not texto:                      # seção que só agrupa subseções
            continue
        for parte, trecho in enumerate(_dividir(texto, TAMANHO_MAXIMO_CHUNK), start=1):
            assinatura = hashlib.sha1(trecho.encode("utf-8")).hexdigest()[:8]
            chunks.append({
                "id": f"{codigo}§{secao['secao']}#{parte}-{assinatura}",
                "texto": trecho,
                "metadata": {
                    "documento": codigo,
                    "titulo_documento": DOCUMENTOS_BASE[codigo],
                    "secao": secao["secao"],
                    "titulo": secao["titulo"],
                    "referencia": f"{codigo} § {secao['secao']}",
                    "aplicacao": aplicacao,
                },
            })
    return chunks


CHUNKS = [c for codigo, caminho in ARQUIVOS_PDF.items() for c in segmentar_documento(codigo, caminho)]

resumo_chunks = pd.DataFrame([c["metadata"] | {"caracteres": len(c["texto"])} for c in CHUNKS])
print(f"{len(CHUNKS)} chunks · média de {resumo_chunks['caracteres'].mean():.0f} caracteres "
      f"· maior com {resumo_chunks['caracteres'].max()}")
display(resumo_chunks.groupby("documento").agg(chunks=("referencia", "count"),
                                                 aplicacao=("aplicacao", "first")))

In [ ]:
# Exemplo de chunk: o procedimento de conduta em falha de freio da carregadeira
exemplo = next(c for c in CHUNKS if c["metadata"]["referencia"] == "FTM-CAT924H § 5.3")
print(exemplo["metadata"]["referencia"], "—", exemplo["metadata"]["titulo"])
print("-" * 78)
print(exemplo["texto"])

## 5.3 Embeddings e índice vetorial

O índice usa ChromaDB persistente com distância de cosseno. O texto vetorizado inclui o título do documento e da seção — isso ajuda a busca a associar *"conduta e diagnóstico"* ao contexto de *"sistema de freios da carregadeira"*.

A indexação é **idempotente**: o id de cada chunk inclui um hash do conteúdo. Reexecutar a célula não gera novos embeddings; se um documento mudar no repositório, só os chunks alterados são reindexados e os obsoletos são removidos. Toda a base cabe em uma única requisição de embedding.

In [ ]:
DIRETORIO_CHROMA = "chroma_db"
NOME_COLECAO = "apollo_base_tecnica"

cliente_chroma = chromadb.PersistentClient(path=DIRETORIO_CHROMA)
colecao = cliente_chroma.get_or_create_collection(
    name=NOME_COLECAO, metadata={"hnsw:space": "cosine"}
)

ids_atuais = [c["id"] for c in CHUNKS]
obsoletos = sorted(set(colecao.get()["ids"]) - set(ids_atuais))
if obsoletos:
    colecao.delete(ids=obsoletos)

ja_indexados = set(colecao.get(ids=ids_atuais)["ids"])
novos = [c for c in CHUNKS if c["id"] not in ja_indexados]

if novos:
    vetores = llm.embed(
        [f"{c['metadata']['titulo_documento']} — {c['metadata']['titulo']}\n{c['texto']}" for c in novos],
        tipo="RETRIEVAL_DOCUMENT",
    )
    colecao.upsert(
        ids=[c["id"] for c in novos],
        embeddings=vetores,
        documents=[c["texto"] for c in novos],
        metadatas=[c["metadata"] for c in novos],
    )

print(f"Índice pronto: {colecao.count()} chunks · novos nesta execução: {len(novos)} "
      f"· removidos: {len(obsoletos)}")
print(llm.resumo())

## 5.4 Busca

`buscar_trechos` é o mecanismo de recuperação. Na seção 6 ele será exposto ao agente como a ferramenta `buscar_documentacao_tecnica`.

- **Filtro por aplicação.** Com `ficha_tecnica` informada, a busca considera só a ficha daquele equipamento e os documentos gerais.
- **Cache de consultas.** O vetor de cada pergunta é guardado em memória; repetir a mesma pergunta não gera nova requisição.

In [ ]:
@lru_cache(maxsize=256)
def _vetor_da_consulta(pergunta: str) -> tuple[float, ...]:
    return tuple(llm.embed([pergunta], tipo="RETRIEVAL_QUERY")[0])


def buscar_trechos(pergunta: str, ficha_tecnica: str | None = None, k: int = 3) -> list[dict[str, Any]]:
    """Recupera os k trechos mais próximos da pergunta, opcionalmente filtrando por ficha técnica."""
    filtro = {"aplicacao": {"$in": [ficha_tecnica, "geral"]}} if ficha_tecnica else None
    resultado = colecao.query(
        query_embeddings=[list(_vetor_da_consulta(pergunta.strip()))],
        n_results=k,
        where=filtro,
        include=["documents", "metadatas", "distances"],
    )
    return [
        {
            "referencia": meta["referencia"],
            "documento": meta["titulo_documento"],
            "titulo": meta["titulo"],
            "aplicacao": meta["aplicacao"],
            "trecho": texto,
            "distancia": round(distancia, 4),
        }
        for texto, meta, distancia in zip(
            resultado["documents"][0], resultado["metadatas"][0], resultado["distances"][0]
        )
    ]

## 5.5 Verificação da recuperação

Antes de entregar a busca ao agente, conferimos se ela encontra o trecho certo para perguntas típicas da triagem. São cinco consultas, uma requisição de embedding cada:

- **Posição do esperado:** em que lugar do top-3 aparece a seção que responde à pergunta.
- **Filtro respeitado:** quando há ficha técnica, nenhum trecho de outra ficha pode aparecer. Essa verificação é determinística e interrompe o notebook se falhar.

In [ ]:
CONSULTAS_DE_VERIFICACAO = [
    ("alarme TL-214 lança e braço lentos depois que a máquina aquece", "FTM-CAT320", "FTM-CAT320 § 5.2"),
    ("pedal de freio baixo e carregadeira demorando para parar", "FTM-CAT924H", "FTM-CAT924H § 5.2"),
    ("quando é obrigatório bloquear o equipamento", None, "PRO-SEG-001 § 3"),
    ("quantas ocorrências caracterizam falha recorrente", None, "POL-MAN-001 § 6"),
    ("ruído na dianteira do caminhão, quais informações a ordem de serviço precisa ter",
     "FTM-MBCAM", "FTM-MBCAM § 5.1"),
]

linhas = []
for pergunta, ficha, esperada in CONSULTAS_DE_VERIFICACAO:
    trechos = buscar_trechos(pergunta, ficha_tecnica=ficha)
    referencias = [t["referencia"] for t in trechos]
    if ficha:
        fora_do_filtro = [t["referencia"] for t in trechos if t["aplicacao"] not in (ficha, "geral")]
        assert not fora_do_filtro, f"Filtro violado para {ficha}: {fora_do_filtro}"
    linhas.append({
        "pergunta": pergunta[:60] + ("…" if len(pergunta) > 60 else ""),
        "filtro": ficha or "—",
        "esperada": esperada,
        "posição": referencias.index(esperada) + 1 if esperada in referencias else "fora do top-3",
        "top-3 recuperado": " | ".join(referencias),
    })

with pd.option_context("display.max_colwidth", 80):
    display(pd.DataFrame(linhas))

acertos = sum(isinstance(l["posição"], int) for l in linhas)
print(f"Esperada no top-3 em {acertos}/{len(linhas)} consultas · filtro por aplicação respeitado em todas.")
print(llm.resumo())